In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://enhantica.github.io/edi-download/pypi/simple/


# Rietveld fit: CeCoAl₃, POLARIS

Fit CeCoAl₃ to time-of-flight neutron powder data from POLARIS at ISIS: structure and profile,
with a 24-term Chebyshev background.

The project is the catalogue case `pd-neut-tof_cecoal-polaris`, the one the command line and the
app's Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">tof</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-tof_cecoal-polaris', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['cecoal']
experiment = project.experiments['polaris']

Data 'pd-neut-tof_cecoal-polaris' is at data/pd-neut-tof_cecoal-polaris-e1e721892155/pd-neut-tof_cecoal-polaris


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('polaris')

View(html=<598,163 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, time-of-flight calibration and cell

The scale, the DIFC offset and its quadratic term, and the three cell lengths put the peaks in
place and at the right height.

In [6]:
experiment.linked_structures['cecoal'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True
experiment.instrument.calib_d_to_tof_quadratic.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cecoal,cell,,length_a,7.65,,0,30,Å
2,cecoal,cell,,length_b,4.05,,0,30,Å
3,cecoal,cell,,length_c,6.91,,0,30,Å
4,polaris,instrument,,calib_d_to_tof_offset,0,,-inf,inf,µs
5,polaris,instrument,,calib_d_to_tof_quadratic,-1.92194,,-inf,inf,µs/Å²
6,polaris,linked_structure,cecoal,scale,2.00×10⁴,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 6 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.38        84.57
      1      0.073      4.17        11.38   86.5% ↓
      2      0.129      4.13        11.14   2.2% ↓
      3      0.185      4.13        11.13   0.1% ↓
      4      0.314      4.13        11.13   
  done · converged · 4 iters · reduced χ² 11.13 · Rwp 4.13 % · 0.39 s


View(html=<84,013 characters>, height=376)

In [9]:
project.display.pattern('polaris')

View(html=<596,217 characters>, height=480)

### Background

The first twelve polynomial terms (orders 0 to 11) follow the level between the peaks; the
higher ones stay at zero.

In [10]:
for term in list(experiment.background_terms)[:12]:
    term.coef.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 18 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.13        11.17
      1      0.111      2.32         3.51   68.5% ↓
      2      0.167      2.31         3.51   0.1% ↓
      3      0.278      2.31         3.51   
  done · converged · 3 iters · reduced χ² 3.51 · Rwp 2.31 % · 0.35 s


View(html=<95,869 characters>, height=376)

In [12]:
project.display.pattern('polaris')

View(html=<615,921 characters>, height=480)

### Peak widths

In [13]:
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_lorentz_gamma_1.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.31         3.51
      1      0.111      2.31         3.51   0.2% ↓
      2      0.187      2.31         3.51   
      3      0.301      2.31         3.51   
  done · converged · 3 iters · reduced χ² 3.51 · Rwp 2.31 % · 0.38 s


View(html=<98,254 characters>, height=376)

In [15]:
project.display.pattern('polaris')

View(html=<615,816 characters>, height=480)

### Atom positions

The coordinates the symmetry leaves free.

In [16]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 25 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.31         3.51
      1      0.115      2.14         3.00   14.4% ↓
      2      0.175      2.14         3.00   0.1% ↓
      3      0.254      2.14         3.00   
      4      0.369      2.14         3.00   
  done · converged · 4 iters · reduced χ² 3.00 · Rwp 2.14 % · 0.45 s


View(html=<103,400 characters>, height=376)

In [18]:
project.display.pattern('polaris')

View(html=<615,761 characters>, height=480)

### Displacements

In [19]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 30 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.14         3.01
      1      0.115      2.10         2.90   3.6% ↓
      2      0.175      2.10         2.89   0.2% ↓
      3      0.253      2.10         2.89   
      4      0.369      2.10         2.89   
  done · converged · 4 iters · reduced χ² 2.89 · Rwp 2.10 % · 0.45 s


View(html=<108,664 characters>, height=376)

In [21]:
project.display.pattern('polaris')

View(html=<615,622 characters>, height=480)

### Absorption

Absorption weakens the intensities with time of flight, as the displacements do with angle, so
it comes after them, on its own.

In [22]:
experiment.absorption.abscor1.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 31 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.10         2.90
      1      0.115      2.09         2.87   0.9% ↓
      2      0.176      2.09         2.86   0.3% ↓
      3      0.236      2.08         2.86   0.1% ↓
      4      0.317      2.08         2.86   
      5      0.434      2.08         2.86   
  done · converged · 5 iters · reduced χ² 2.86 · Rwp 2.08 % · 0.51 s


View(html=<109,829 characters>, height=376)

In [24]:
project.display.pattern('polaris')

View(html=<615,900 characters>, height=480)

### Al3 occupancy

Last, Al3's occupancy, which moves with its displacement.

In [25]:
structure.atom_sites['Al3'].occupancy.free = True

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 32 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.08         2.86
      1      0.116      2.08         2.85   0.2% ↓
      2      0.196      2.08         2.85   
      3      0.312      2.08         2.85   
  done · converged · 3 iters · reduced χ² 2.85 · Rwp 2.08 % · 0.39 s


View(html=<110,857 characters>, height=376)

In [27]:
project.display.pattern('polaris')

View(html=<615,940 characters>, height=480)

In [28]:
project.display.fit.correlations()

,1,2,3,4,5
1 cecoal.cell.length_b,,,,,
2 cecoal.cell.length_c,0.94,,,,
3 polaris.instrument.calib_d_to_tof_quadratic,-0.96,-0.95,,,
4 polaris.background.1.coef,0.00,0.00,-0.00,,
5 polaris.background.2.coef,0.00,0.00,-0.00,0.97,


In [29]:
project.display.structure('cecoal')

View(html=<228,739 characters>, height=660)

## Save the project

In [30]:
project.save_as('pd-neut-tof_cecoal-polaris')
project.report.save_html('pd-neut-tof_cecoal-polaris.html')

Saving project to 'pd-neut-tof_cecoal-polaris'
Saving report to 'pd-neut-tof_cecoal-polaris.html'
